In [1]:
# ============================================
# CODING EXPERT + CHAT BOT - Free Colab Version
# Optimized for T4 GPU (16GB VRAM)
# ============================================

# 1. INSTALL DEPENDENCIES
%pip install -q transformers datasets accelerate peft bitsandbytes trl

# 2. IMPORTS
import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    BitsAndBytesConfig
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer
from datasets import load_dataset, concatenate_datasets
import random

# 3. MEMORY-EFFICIENT CONFIG FOR T4
MODEL_NAME = "codellama/CodeLlama-7b-Instruct-hf"  # Code + chat capable

# 4-bit quantization config (aggressive for T4)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,  # float16 for T4 (no bfloat16)
    bnb_4bit_use_double_quant=True,
)

# LoRA config (slightly smaller for T4 stability)
lora_config = LoraConfig(
    r=32,              # Reduced from 64 for memory
    lora_alpha=64,     # 2x rank (good rule of thumb)
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

# 4. LOAD MODEL
print("🚀 Loading CodeLlama 7B (4-bit)...")
print("This takes ~2 minutes on Colab...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"  # Important for training

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
)
model.config.use_cache = False  # Disable cache for training

# Prepare for training
model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, lora_config)

# Show trainable parameters
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f"✅ Trainable params: {trainable_params:,} ({100*trainable_params/total_params:.2f}%)")

# 5. LOAD & MIX DATASETS
print("\n📚 Loading datasets...")

# CODE DATASET (70%)
print("Loading code dataset...")
code_dataset = load_dataset("iamtarun/python_code_instructions_18k_alpaca", split="train")
# Take 15K examples
code_dataset = code_dataset.shuffle(seed=42).select(range(min(15000, len(code_dataset))))

# GENERAL CHAT DATASET (30%)
print("Loading chat dataset...")
chat_dataset = load_dataset("mlabonne/orpo-dpo-mix-40k", split="train")
# Take 6K examples (30% of total ~21K)
chat_dataset = chat_dataset.shuffle(seed=42).select(range(min(6000, len(chat_dataset))))

print(f"Code examples: {len(code_dataset)}")
print(f"Chat examples: {len(chat_dataset)}")

# 6. FORMAT FUNCTIONS
def format_code(example):
    """Format code instructions"""
    instruction = example.get('prompt', example.get('instruction', ''))
    input_text = example.get('input', '')
    output = example.get('completion', example.get('output', ''))

    if input_text:
        prompt = f"{instruction}\n\nInput: {input_text}"
    else:
        prompt = instruction

    # CodeLlama chat format
    return f"""[INST] {prompt.strip()} [/INST] {output.strip()}"""

def format_chat(example):
    """Format general chat"""
    prompt = example.get('prompt', example.get('question', example.get('instruction', '')))
    response = example.get('chosen', example.get('response', example.get('answer', example.get('output', ''))))

    if isinstance(response, list):
        response = response[0] if response else ""

    return f"""[INST] {prompt.strip()} [/INST] {str(response).strip()}"""

# Apply formatting
print("Formatting datasets...")
code_formatted = code_dataset.map(
    lambda x: {"text": format_code(x)},
    remove_columns=code_dataset.column_names
)

chat_formatted = chat_dataset.map(
    lambda x: {"text": format_chat(x)},
    remove_columns=chat_dataset.column_names
)

# Combine and shuffle
full_dataset = concatenate_datasets([code_formatted, chat_formatted])
full_dataset = full_dataset.shuffle(seed=42)

print(f"\n✅ Total training examples: {len(full_dataset)}")
print(f"Sample: {full_dataset[0]['text'][:200]}...")

# 7. TRAINING ARGUMENTS (T4-OPTIMIZED)
training_args = TrainingArguments(
    output_dir="./codellama-coding-chat",
    num_train_epochs=1,                    # 1 epoch for speed (increase to 2-3 for quality)
    per_device_train_batch_size=1,       # T4 can only handle 1 with 2048 length
    gradient_accumulation_steps=8,         # Effective batch = 8
    optim="paged_adamw_8bit",
    learning_rate=2e-4,
    max_grad_norm=0.3,
    warmup_ratio=0.05,                   # 5% warmup
    lr_scheduler_type="cosine",
    logging_steps=25,
    save_strategy="steps",
    save_steps=500,
    fp16=True,
    gradient_checkpointing=True,
    report_to="none",
)

# 8. TRAIN
print("\n🔥 Starting training...")
print("This will take ~2-3 hours on T4 for 1000 steps...")

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=full_dataset,
    max_seq_length=1024,                 # Reduced for T4 memory
    args=training_args,
    dataset_text_field="text",
)

trainer.train()

# 9. SAVE MODEL
print("\n💾 Saving model...")
model.save_pretrained("./codellama-coding-chat-final")
tokenizer.save_pretrained("./codellama-coding-chat-final")

print("✅ Training complete! Model saved.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 557.0/557.0 kB 15.8 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 23.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 528.8/528.8 kB 42.2 MB/s eta 0:00:00


KeyboardInterrupt: 